# 1. Config & Imports

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Config:
    audio_root : str = ""
    features_csv = str = "pcd_pcdd_features.csv"

    # audio settings
    sr : int = 22050
    analysis_duration : float | None = None # considered full clip
    fmin_hz : float = 80.0
    fmax_hz : float = 800.0
    n_pitch_classes : int = 12 # PCD/PCDD resolution (12 = standard chroma-like)

    random_seed : int = 42
    AUDIO_EXTENSIONS = (".wav", ".mp3", ".flac", ".m4a")
cfg = Config()


In [5]:
import os
import re
import warnings

import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
np.random.seed(CFG.random_seed)

ModuleNotFoundError: No module named 'pandas'

### Helper functions

In [12]:
# Raga IDs for 300 Hindustani Classical Music clips (30 ragas x 10 songs)
ID_TO_RAGA = {
    'f7fddfc0-8c1d-4dd2-90d5-5d51a99d61f8 2': 'Rāgēśrī',
    'f6432fec-e9c2-4b09-9e73-c46086cbd8ea': 'Ahira bhairav',
    '290674e0-d94c-41c1-ad99-f65fa22a1447': 'Madhuvanti',
    '0b3bbf97-0ec3-41da-add4-722d87329ec3': 'Madhukauns',
    'a9ee554f-f146-43e9-b2d0-9bb31fd4b57d': 'Kēdār',
    'd9c603fa-875f-4b84-b851-c6a345427898': 'Ābhōgī',
    '1e7de02f-e77f-405a-a033-f31117aaf955': 'Bhairav',
    '3eb7ba30-4b94-432e-9618-875ee57e01ab': 'Mārvā',
    '46997b02-f09c-4969-8138-4e1861f61967': 'Śrī',
    '54c4214c-05b9-4acc-8f77-6d5786e43a2e': 'Mārūbihāg',
    'a99e07d5-20a0-467b-8dcd-aa5a095177fd': 'Lalit',
    '62b79291-73b0-4c77-a353-0f8bc6ed8362': 'Dēś',
    '118401e7-8de8-4d81-9d8e-8070889e3fa8': 'Darbāri',
    '7591faad-e68a-4550-b675-8082842c6056': 'Bāgēśrī',
    '93c73081-bdf8-4eca-b325-d736b71e9b4b': 'Khamāj',
    'ba3242d0-8d40-4d93-865e-d2e2497ea2a8': 'Gauḍ malhār',
    'ecd04ceb-b46c-47fc-9045-84ac9160e527': 'Basant',
    '6f13484e-6fdd-402d-baf3-3835835454d0': 'Tōḍī',
    'fa28470c-d413-44c7-94da-181f530cbfdd': 'Puriyā dhanaśrī',
    'f7fddfc0-8c1d-4dd2-90d5-5d51a99d61f8': 'Rāgēśrī',
    '40dbe1db-858f-4366-bef8-9076eb67340d': 'Śuddh sāraṅg',
    '1b05a564-059f-445b-b325-cf26318367e3': 'Miyān malhār',
    'b143adaa-f1a6-4de4-8985-a5bd35e96279': 'Bairāgi',
    '64e5fb9e-5569-4e80-8e6c-f543af9469c7': 'Mālkauns',
    '2ed9379f-14c9-49af-8e4d-f9b63e96801f': 'Alahaiyā bilāval',
    'e771a74d-545d-41d5-816b-43403a818b0c': 'Bhūp',
    '063ea5a0-23b1-4bb5-8537-3d924fe8ebb3': 'Jōg',
    'a7d98897-e2fe-4d75-b9dc-e5b4dc88e1c6': 'Bihāg',
    '48b37bed-e847-4882-8a01-5c721e07f07d': 'Yaman kalyāṇ',
    '595de771-fcc2-414c-9df6-ae4ffd898549': 'Hamsadhvāni',
    'dd59147d-8775-44ff-a36b-0d9f15b31319': 'Bilāsakhānī tōḍī',
}

print(f"Loaded {len(ID_TO_RAGA)} raga id mappings "
      f"({len(set(ID_TO_RAGA.values()))} unique raga labels)")


def clean_uuid(raw_name: str) -> str:
    '''Normalize a folder name to its canonical uuid string.'''
    name = raw_name.strip()
    name = re.sub(r"\s*\(\d+\)$", "", name)   # trailing " (1)", " (2)"
    name = re.sub(r"\s+\d+$", "", name)       # trailing " 2", " 3"
    return name.lower()


def build_normalized_lookup(id_to_raga: dict) -> dict:
    '''Collapse duplicate-suffixed uuid keys into one clean lookup table.'''
    return {clean_uuid(k): v for k, v in id_to_raga.items()}


def find_audio_files_with_labels(root_dir: str, id_to_raga: dict) -> list[tuple[str, str, str]]:
    '''Walk `root_dir` and return (filepath, raga_label, artist) for every
    audio file whose parent uuid folder is present in `id_to_raga`.
    '''
    lookup = build_normalized_lookup(id_to_raga)
    items = []

    for uuid_folder in sorted(os.listdir(root_dir)):
        uuid_path = os.path.join(root_dir, uuid_folder)
        if not os.path.isdir(uuid_path):
            continue

        raga_label = lookup.get(clean_uuid(uuid_folder))
        if raga_label is None:
            print(f"WARNING: no raga mapping for uuid folder '{uuid_folder}', skipping")
            continue

        for dirpath, _, filenames in os.walk(uuid_path):
            rel_path = os.path.relpath(dirpath, uuid_path)
            artist = rel_path.split(os.sep)[0] if rel_path != "." else "unknown"

            for fname in filenames:
                if fname.lower().endswith(cfg.AUDIO_EXTENSIONS):
                    items.append((os.path.join(dirpath, fname), raga_label, artist))

    return items

Loaded 31 raga id mappings (30 unique raga labels)


# 2. Feature Extraction

### 2.1 Tonic estimation

In [6]:
# "Cents" is just a unit for measuring musical pitch on a logarithmic scale
# 1200 cents = exactly 1 octave --> 100 cents = 1 semitone (12 semitones in an octave)
def estimate_tonic(audio : np.ndarray, sr : int, low_freq : float = cfg.fmin_hz, high_freq : float = cfg.fmax_hz):
    pitch_track, is_voiced, _ = librosa.pyin(
        audio, fmin=low_freq, fmax=high_freq, sr=sr, frame_length=2048
    )
    # keep only the pitch estimates where a clear pitch was actually detected
    if is_voiced is not None:
        clean_pitches = pitch_track[is_voiced]
    else:
        clean_pitches = pitch_track[~np.isnan(pitch_track)]
    clean_pitches = clean_pitches[~np.isnan(clean_pitches)]

    if len(clean_pitches) == 0:
        return 220.0  # no pitch detected in this clip, just return a default

    # convert to cents
    cents = 1200*np.log2(clean_pitches / low_freq)
    cents_in_one_octave = np.mod(cents, 1200)
    counts, bin_edges = np.histogram(cents_in_one_octave, bins=120, range=(0,1200))
    tallest_bin_position = bin_edges[np.argmax(counts)]

    # convert that cents value back into a frequency in Hz
    tonic_hz = low_freq * (2 ** (tallest_bin_position / 1200))
    return float(tonic_hz)

### 2.2 Note segmentation

In [8]:
def extract_note_sequence(y: np.ndarray, sr: int, tonic_hz: float,
                           n_pitch_classes: int = cfg.n_pitch_classes,
                           fmin: float = cfg.fmin_hz, fmax: float = 1000.0) -> list[int]:
    '''Segment audio into notes via onset detection, and return the ordered
    sequence of tonic-relative pitch-class labels (one per note).
    '''
    f0, voiced_flag, _ = librosa.pyin(y, fmin=fmin, fmax=fmax, sr=sr, frame_length=2048)
    times = librosa.times_like(f0, sr=sr)

    onset_times = librosa.onset.onset_detect(y=y, sr=sr, units="time")
    duration = times[-1] if len(times) else len(y) / sr
    boundaries = np.unique(np.concatenate([[0.0], onset_times, [duration]]))

    note_classes = []
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        mask = (times >= start) & (times < end)
        seg_voiced = f0[mask & (voiced_flag == True)] if voiced_flag is not None else f0[mask]
        seg_voiced = seg_voiced[~np.isnan(seg_voiced)]
        seg_voiced = seg_voiced[seg_voiced > 0]
        if len(seg_voiced) == 0:
            continue  # silent / unvoiced segment — skip, no note here

        note_pitch = np.median(seg_voiced)
        cents_rel = 1200 * np.log2(note_pitch / tonic_hz)
        pitch_class = int(np.mod(cents_rel, 1200) / (1200 / n_pitch_classes)) % n_pitch_classes
        note_classes.append(pitch_class)

    return note_classes

### 2.3 PCD (Pitch Class Distribution)

In [9]:
def compute_pcd(note_classes : list[int], n_pitch_classes: int = cfg.n_pitch_classes) -> np.ndarray:
    hist = np.zeros(n_pitch_classes)
    for pc in note_classes:
        hist[pc] += 1
    if hist.sum() > 0:
        hist /= hist.sum()
    return hist

### 2.3 PCDD (Pitch Class Dyads Distribution)

In [11]:
def compute_pcdd(note_classes: list[int], n_pitch_classes: int = cfg.n_pitch_classes) -> np.ndarray:
    '''Normalized, flattened histogram of consecutive note-pitch-class pairs (dyads).'''
    hist = np.zeros((n_pitch_classes, n_pitch_classes))
    for a, b in zip(note_classes[:-1], note_classes[1:]):
        hist[a, b] += 1
    flat = hist.flatten()
    return flat / flat.sum() if flat.sum() > 0 else flat

# 3. Build Feature Table

In [13]:
def process_file(filepath: str, raga_label: str, artist: str,
                  sr_target: int = cfg.sr,
                  duration: float | None = cfg.analysis_duration) -> dict:
    '''Load one clip and compute its PCD + PCDD feature row.'''
    y, sr = librosa.load(filepath, sr=sr_target, duration=duration, mono=True)
    y, _ = librosa.effects.trim(y, top_db=30)

    tonic_hz = estimate_tonic(y, sr)
    notes = extract_note_sequence(y, sr, tonic_hz)
    pcd = compute_pcd(notes)
    pcdd = compute_pcdd(notes)

    row = {"filepath": filepath, "artist": artist, "raga": raga_label,
           "tonic_hz": tonic_hz, "n_notes": len(notes)}
    row.update({f"pcd_{i}": v for i, v in enumerate(pcd)})
    row.update({f"pcdd_{i}": v for i, v in enumerate(pcdd)})
    return row


def build_feature_table(root: str, id_to_raga: dict, out_csv: str) -> pd.DataFrame:
    '''Extract PCD + PCDD features for every labeled audio file and cache to CSV.'''
    labeled_files = find_audio_files_with_labels(root, id_to_raga)
    print(f"Found {len(labeled_files)} audio files with a known raga label under {root}")

    rows = []
    for filepath, raga_label, artist in tqdm(labeled_files, desc="Extracting features"):
        try:
            rows.append(process_file(filepath, raga_label, artist))
        except Exception as e:
            print(f"FAILED  {filepath}: {e}")

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)

    n_ragas = df["raga"].nunique() if len(df) else 0
    print(f"\nSaved {len(df)} feature rows to {out_csv}")
    print(f"Ragas represented: {n_ragas} / {len(set(id_to_raga.values()))} in the mapping")
    return df

In [ ]:
if os.path.exists(cfg.features_csv):
    print(f"Loading cached features from {cfg.features_csv}")
    features_df = pd.read_csv(cfg.features_csv)
else:
    features_df = build_feature_table(cfg.audio_root, ID_TO_RAGA, cfg.features_csv)

features_df.head()

# 4. EDA

In [ ]:
print(f"Total clips: {len(features_df)}")
print(f"Unique ragas: {features_df['raga'].nunique()}")
print(f"Unique artists: {features_df['artist'].nunique()}")
print(f"Median notes detected per clip: {features_df['n_notes'].median():.0f}")

fig, ax = plt.subplots(figsize=(10, 6))
features_df["raga"].value_counts().sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Number of clips")
ax.set_title("Clips per raga — check this is roughly balanced")
plt.tight_layout()
plt.show()

In [ ]:
pcd_cols = [c for c in features_df.columns if c.startswith("pcd_")]
pcdd_cols = [c for c in features_df.columns if c.startswith("pcdd_")]

fig, axes = plt.subplots(2, 1, figsize=(10, 6))
example = features_df.sample(1, random_state=cfg.random_seed).iloc[0]

axes[0].bar(range(len(pcd_cols)), example[pcd_cols].values, color="darkslateblue")
axes[0].set_title(f"PCD — {example['raga']} ({example['artist']})")
axes[0].set_xlabel("Pitch class")

axes[1].bar(range(len(pcdd_cols)), example[pcdd_cols].values, color="darkorange", width=1.0)
axes[1].set_title("PCDD (flattened pitch-class-pair histogram)")
axes[1].set_xlabel("Dyad index (pitch_class_a * n_pitch_classes + pitch_class_b)")

plt.tight_layout()
plt.show()